# UI Element Detector — Kaggle training (no Drive needed)

Two stages, same split and seed:

1. **YOLO26n** finds element boxes on the full screenshot.
2. **DenseNet121, Xception41, SE-ResNet50, PyramidNet-110, Residual Attention Network** classify each detected crop.

Everything stays inside Kaggle: code is cloned from GitHub, the data is read from whatever you
uploaded (dataset or direct upload), and all results end up in `/kaggle/working`.

## Before you run

1. **Settings → Accelerator: GPU T4 x2** (or *GPU P100* — faster).
2. **Settings → Internet: On** (needed for the GitHub clone and the ImageNet weights).
3. Have the data somewhere Kaggle can see it — any of these works, cell 3 finds it itself:
   * a **Dataset** with `ui-data.zip` (add it with *Add Input*),
   * `ui-data.zip` uploaded straight into the notebook (`/kaggle/working`),
   * a dataset that is **already extracted** (a folder containing `data.yaml` + `images/`).

Phone-verify the account first, otherwise the GPU and Internet switches stay locked.

Limits: **12 h per session, 30 h GPU per week**. The full run is ~12–15 h, so plan on 2–3 sessions.


In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Code

Nothing to do manually — this cell fetches the code itself.

* normal case: `git clone` from GitHub into `/kaggle/working/project` (needs **Internet: On**);
* if the clone fails (no internet): it unpacks `ui-element-detector-code.zip` from `/kaggle/input`
  or `/kaggle/working`, so you can also just upload that zip as a dataset.

In [ ]:
import glob, os, pathlib, subprocess, zipfile

PROJECT = pathlib.Path('/kaggle/working/project')
REPO_URL = 'https://github.com/AnaraHayat/ui-Element-detector.git'

if not (PROJECT / 'scripts').is_dir():
    print('cloning', REPO_URL)
    r = subprocess.run(f'git clone -q {REPO_URL} {PROJECT}', shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print('clone failed:', (r.stderr or '').strip()[:200])
        zips = (sorted(glob.glob('/kaggle/input/**/*code*.zip', recursive=True)) +
                sorted(glob.glob('/kaggle/working/*code*.zip')))
        assert zips, ('cannot get the code: enable Internet in Session options, '
                      'or upload ui-element-detector-code.zip as a dataset')
        print('unpacking code from', zips[0])
        zipfile.ZipFile(zips[0]).extractall(PROJECT)

os.chdir(PROJECT)
assert (PROJECT / 'scripts').is_dir(), f'code not found in {PROJECT}'
print('code ready:', os.getcwd())
print(sorted(p.name for p in PROJECT.iterdir())[:8])

!pip install -q timm ultralytics

## 2. Find and connect the data

This cell looks in this order and stops at the first hit:

1. `project/data/yolo/data.yaml` already set up (second run in the same session) → nothing to do.
2. a `ui-data*.zip` under `/kaggle/input` (dataset) or `/kaggle/working` (direct upload) → unpacked to `project/data/yolo`.
3. an already-extracted dataset containing `data.yaml` + `images/train` → linked, not copied, so no extra disk.

It then writes a portable `data.yaml` (relative paths, no machine-specific `path:` key).

In [ ]:
import glob, os, pathlib, shutil, yaml

PROJECT = pathlib.Path('/kaggle/working/project')
YOLO = PROJECT / 'data' / 'yolo'          # where every script expects the dataset
YOLO.mkdir(parents=True, exist_ok=True)
yaml_file = YOLO / 'data.yaml'


def has_images(root: pathlib.Path) -> bool:
    return (root / 'images' / 'train').is_dir() and any((root / 'images' / 'train').glob('*.jpg'))


def find_root() -> pathlib.Path | None:
    """A folder that already contains data.yaml next to images/ and labels/."""
    for pattern in ('/kaggle/input/**/data.yaml', '/kaggle/working/**/data.yaml'):
        for y in glob.glob(pattern, recursive=True):
            root = pathlib.Path(y).parent
            if has_images(root):
                return root
    return None


def find_zip() -> pathlib.Path | None:
    for pattern in ('/kaggle/input/**/*data*.zip', '/kaggle/working/*data*.zip'):
        for z in sorted(glob.glob(pattern, recursive=True)):
            p = pathlib.Path(z)
            if 'code' not in p.name.lower():      # skip ui-element-detector-code.zip
                return p
    return None


if yaml_file.exists():
    print('dataset already prepared:', YOLO)
else:
    root = find_root()
    if root is not None:
        print('using already-extracted dataset:', root)
        for sub in ('images', 'labels', 'class_index'):
            src, dst = root / sub, YOLO / sub
            if src.exists() and not dst.exists():
                try:
                    os.symlink(src, dst, target_is_directory=True)   # link, do not copy 7.8 GB
                except OSError:
                    shutil.copytree(src, dst)
        cfg = yaml.safe_load((root / 'data.yaml').read_text())
        names = cfg['names']
        out = {'train': 'images/train', 'val': 'images/val', 'test': 'images/test',
               'nc': len(names), 'names': names}
        yaml_file.write_text(yaml.safe_dump(out, sort_keys=False))
        print('linked, data.yaml written at', yaml_file)
    else:
        zip_path = find_zip()
        assert zip_path is not None, (
            'no data found. Upload ui-data.zip or add it as a Kaggle dataset '
            '(Add Input -> Your datasets), then re-run this cell.')
        print('unpacking', zip_path, round(zip_path.stat().st_size / 1e9, 2), 'GB')
        with __import__('zipfile').ZipFile(zip_path) as zf:
            zf.extractall(PROJECT / 'data')
        if not yaml_file.exists():                     # zip with a different root folder
            inner = find_root()
            assert inner is not None, f'unpacked {zip_path} but found no data.yaml + images/'
            for sub in ('images', 'labels', 'class_index'):
                if (inner / sub).exists() and not (YOLO / sub).exists():
                    os.symlink(inner / sub, YOLO / sub, target_is_directory=True)
            shutil.copy(inner / 'data.yaml', yaml_file)
        cfg = yaml.safe_load(yaml_file.read_text())
        cfg.pop('path', None)                          # absolute Windows path breaks on Kaggle
        yaml_file.write_text(yaml.safe_dump(cfg, sort_keys=False))
        print('unpacked to', YOLO)

print(yaml_file.read_text())
for split in ('train', 'val', 'test'):
    n = len(list((YOLO / 'images' / split).glob('*.jpg')))
    print(f'{split:>5}: {n:,} images')
assert len(list((YOLO / 'images' / 'train').glob('*.jpg'))) > 40000, 'training split looks wrong'

## 3. Dataset sanity check

In [ ]:
from ultralytics.data.utils import check_det_dataset
d = check_det_dataset(str(pathlib.Path('data/yolo/data.yaml')))
print('train:', d['train'], '| nc:', d['nc'])
print(list(d['names'].values()))

## 4. Smoke test (a few minutes)

Downloads the three ImageNet checkpoints and trains densenet121 for 1 epoch on 500 crops.
If this passes, GPU, data and weights are all fine. **Do not skip it.**

In [ ]:
!python scripts/models/check_models.py --size 128
!python scripts/training/train_cnn.py --model densenet121 --epochs 1 --max-crops 500 --val-per-class 20 --batch-size 32 --out-suffix _smoke

## 5. Five classifiers

Checkpoints are copied to `/kaggle/working/saved`, and any model whose checkpoint is already
there is skipped — so a new session only trains what is missing.

To split the work across sessions, edit `MODELS`, e.g. session 1:
`MODELS = ['densenet121', 'xception41', 'seresnet50']`.

In [ ]:
import glob, os, shutil
MODELS = ['densenet121', 'xception41', 'seresnet50', 'pyramidnet110', 'residual_attention']
SAVE = '/kaggle/working/saved'
os.makedirs(SAVE, exist_ok=True)

# a previous session's checkpoints, added via Add Input -> Your work -> Notebook output,
# are copied back so already-trained models are skipped
restored = []
for p in glob.glob('/kaggle/input/**/models_*_best.pt', recursive=True):
    dst = os.path.join(SAVE, os.path.basename(p))
    if not os.path.exists(dst):
        shutil.copy(p, dst); restored.append(os.path.basename(p))
print('restored from earlier sessions:', restored or 'none')

for m in MODELS:
    if os.path.exists(f'{SAVE}/models_{m}_best.pt'):
        print('already trained, skipping:', m)
        continue
    print('=' * 70); print('TRAINING', m)
    !python scripts/training/train_cnn.py --model {m}
    shutil.copy(f'results/{m}/best.pt', f'{SAVE}/models_{m}_best.pt')

## 6. Detector (YOLO26n)

In [ ]:
import glob, os, shutil
SAVE = '/kaggle/working/saved'
os.makedirs(SAVE, exist_ok=True)
for p in glob.glob('/kaggle/input/**/models_yolo26n_best.pt', recursive=True):
    dst = f'{SAVE}/models_yolo26n_best.pt'
    if not os.path.exists(dst):
        shutil.copy(p, dst)

if os.path.exists(f'{SAVE}/models_yolo26n_best.pt'):
    print('already trained, skipping: yolo26n')
else:
    !python scripts/training/train_yolo.py --model yolo26n
    shutil.copy('results/yolo26n/weights/best.pt', f'{SAVE}/models_yolo26n_best.pt')

## 7. Package the results

`/kaggle/working` is kept when you **Save Version** (top right → *Quick Save* / *Save & Run All*).
Then open the notebook's **Output** tab and download `ui-results.zip`.

Do a *Quick Save* after each model finishes — it costs nothing and protects against the 12 h cutoff.

In [ ]:
import shutil, os
shutil.make_archive('/kaggle/working/ui-results', 'zip', 'results')
print('saved:', round(os.path.getsize('/kaggle/working/ui-results.zip') / 1e6, 1), 'MB')
for f in sorted(os.listdir('/kaggle/working/saved')):
    print(' ', f)